In [1]:
# ============================================================
# LOGISTIC REGRESSION MODEL
# 5-FOLD STRATIFIED CROSS VALIDATION
# PHISHING WEBSITE DETECTION
# ============================================================

import pandas as pd
import numpy as np
import re
from urllib.parse import urlparse

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# STEP 1: LOAD CLEANED DATASET
# ============================================================

print("=" * 70)
print("LOGISTIC REGRESSION - 5-FOLD CROSS VALIDATION")
print("=" * 70)

df = pd.read_csv('phishing_website_cleaned.csv')

print("\nDataset shape:", df.shape)

# ============================================================
# STEP 2: DEFINE EXACT SAME URL FEATURES
# ============================================================

URL_FEATURES = [

    # --------------------------------------------------------
    # URL CHARACTER FEATURES
    # --------------------------------------------------------
    'qty_dot_url',
    'qty_hyphen_url',
    'qty_underline_url',
    'qty_slash_url',
    'qty_questionmark_url',
    'qty_equal_url',
    'qty_at_url',
    'qty_and_url',
    'qty_exclamation_url',
    'qty_space_url',
    'qty_tilde_url',
    'qty_comma_url',
    'qty_plus_url',
    'qty_asterisk_url',
    'qty_hashtag_url',
    'qty_dollar_url',
    'qty_percent_url',
    'length_url',

    # --------------------------------------------------------
    # DOMAIN CHARACTER FEATURES
    # --------------------------------------------------------
    'qty_dot_domain',
    'qty_hyphen_domain',
    'qty_underline_domain',
    'qty_slash_domain',
    'qty_questionmark_domain',
    'qty_equal_domain',
    'qty_at_domain',
    'qty_and_domain',
    'qty_exclamation_domain',
    'qty_space_domain',
    'qty_tilde_domain',
    'qty_comma_domain',
    'qty_plus_domain',
    'qty_asterisk_domain',
    'qty_hashtag_domain',
    'qty_dollar_domain',
    'qty_percent_domain',

    # --------------------------------------------------------
    # DOMAIN STATISTICS
    # --------------------------------------------------------
    'qty_vowels_domain',
    'domain_length',
    'domain_in_ip',
    'server_client_domain',
    'email_in_url',

    # --------------------------------------------------------
    # SECURITY / URL FEATURES
    # --------------------------------------------------------
    'tls_ssl_certificate',
    'url_shortened'
]

# ============================================================
# STEP 3: SELECT AVAILABLE FEATURES
# ============================================================

available_features = [
    col for col in URL_FEATURES
    if col in df.columns
]

X = df[available_features].copy()
y = df['phishing'].copy()

print("\nFeatures before removing constants:", X.shape[1])

# ============================================================
# STEP 4: REMOVE CONSTANT FEATURES
# ============================================================

constant_features = [
    col for col in X.columns
    if X[col].nunique() <= 1
]

if constant_features:

    print("\nRemoving constant features:")

    for col in constant_features:
        print("-", col)

    X = X.drop(columns=constant_features)

print("\nFeatures after removing constants:", X.shape[1])

print("\nFinal feature list:")
for i, feature in enumerate(X.columns, start=1):
    print(f"{i:2d}. {feature}")

# ============================================================
# STEP 5: CHECK DATA
# ============================================================

print("\n" + "=" * 70)
print("DATA CHECK")
print("=" * 70)

print("\nX shape:", X.shape)
print("y shape:", y.shape)

print("\nClass distribution:")
print(y.value_counts())

print("\nClass distribution (%):")
print(y.value_counts(normalize=True) * 100)

# ============================================================
# STEP 6: CREATE LOGISTIC REGRESSION PIPELINE
# ============================================================
#
# StandardScaler is placed INSIDE the Pipeline.
#
# This is important because during each CV fold:
#
#     Training fold -> fit scaler
#     Training fold -> transform
#     Validation fold -> transform only
#
# Therefore, there is NO data leakage.
# ============================================================

logistic_pipeline = Pipeline([
    ('scaler', StandardScaler()),

    ('logistic_regression', LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# ============================================================
# STEP 7: DEFINE 5-FOLD STRATIFIED CROSS VALIDATION
# ============================================================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("\n" + "=" * 70)
print("5-FOLD STRATIFIED CROSS VALIDATION")
print("=" * 70)

print("\nNumber of folds:", skf.n_splits)
print("Shuffle       :", skf.shuffle)
print("Random state  :", skf.random_state)

# ============================================================
# STEP 8: DEFINE EVALUATION METRICS
# ============================================================

scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1'
}

# ============================================================
# STEP 9: PERFORM 5-FOLD CROSS VALIDATION
# ============================================================

cv_results = cross_validate(
    logistic_pipeline,
    X,
    y,
    cv=skf,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

# ============================================================
# STEP 10: EXTRACT FOLD RESULTS
# ============================================================

fold_accuracy = cv_results['test_accuracy']
fold_precision = cv_results['test_precision']
fold_recall = cv_results['test_recall']
fold_f1 = cv_results['test_f1']

# ============================================================
# STEP 11: DISPLAY INDIVIDUAL FOLD RESULTS
# ============================================================

print("\n" + "=" * 70)
print("INDIVIDUAL FOLD RESULTS")
print("=" * 70)

for fold in range(5):

    print(f"\nFold {fold + 1}")
    print("-" * 40)

    print(
        f"Accuracy : {fold_accuracy[fold] * 100:.2f}%"
    )

    print(
        f"Precision: {fold_precision[fold]:.4f}"
    )

    print(
        f"Recall   : {fold_recall[fold]:.4f}"
    )

    print(
        f"F1-Score : {fold_f1[fold]:.4f}"
    )

# ============================================================
# STEP 12: CALCULATE MEAN AND STANDARD DEVIATION
# ============================================================

accuracy_mean = np.mean(fold_accuracy)
accuracy_std = np.std(fold_accuracy)

precision_mean = np.mean(fold_precision)
precision_std = np.std(fold_precision)

recall_mean = np.mean(fold_recall)
recall_std = np.std(fold_recall)

f1_mean = np.mean(fold_f1)
f1_std = np.std(fold_f1)

# ============================================================
# STEP 13: DISPLAY FINAL CROSS-VALIDATION RESULTS
# ============================================================

print("\n" + "=" * 70)
print("5-FOLD CROSS VALIDATION RESULTS")
print("=" * 70)

print(
    f"\nAccuracy : {accuracy_mean * 100:.2f}% "
    f"(± {accuracy_std * 100:.2f}%)"
)

print(
    f"Precision: {precision_mean:.4f} "
    f"(± {precision_std:.4f})"
)

print(
    f"Recall   : {recall_mean:.4f} "
    f"(± {recall_std:.4f})"
)

print(
    f"F1-Score : {f1_mean:.4f} "
    f"(± {f1_std:.4f})"
)

# ============================================================
# STEP 14: TRAIN FINAL MODEL ON FULL DATASET
# ============================================================
#
# After cross-validation, train one final model using
# the complete dataset.
#
# This model can then be used for live URL prediction.
# ============================================================

print("\n" + "=" * 70)
print("TRAINING FINAL LOGISTIC REGRESSION MODEL")
print("=" * 70)

logistic_pipeline.fit(X, y)

print("\n✓ Final Logistic Regression model trained successfully!")

# ============================================================
# STEP 15: LIVE URL FEATURE EXTRACTION
# ============================================================

def extract_url_features(url):

    url = url.strip()

    if url.startswith('[') and '](' in url:
        url = url.split('](', 1)[1].rstrip(')')

    if not url.startswith(('http://', 'https://')):
        url = 'http://' + url

    parsed = urlparse(url)

    full_url = url

    domain = parsed.netloc.split(':')[0]

    features = {}

    # --------------------------------------------------------
    # URL CHARACTER FEATURES
    # --------------------------------------------------------

    features['qty_dot_url'] = full_url.count('.')
    features['qty_hyphen_url'] = full_url.count('-')
    features['qty_underline_url'] = full_url.count('_')
    features['qty_slash_url'] = full_url.count('/')
    features['qty_questionmark_url'] = full_url.count('?')
    features['qty_equal_url'] = full_url.count('=')
    features['qty_at_url'] = full_url.count('@')
    features['qty_and_url'] = full_url.count('&')
    features['qty_exclamation_url'] = full_url.count('!')
    features['qty_space_url'] = full_url.count(' ')
    features['qty_tilde_url'] = full_url.count('~')
    features['qty_comma_url'] = full_url.count(',')
    features['qty_plus_url'] = full_url.count('+')
    features['qty_asterisk_url'] = full_url.count('*')
    features['qty_hashtag_url'] = full_url.count('#')
    features['qty_dollar_url'] = full_url.count('$')
    features['qty_percent_url'] = full_url.count('%')

    features['length_url'] = len(full_url)

    # --------------------------------------------------------
    # DOMAIN CHARACTER FEATURES
    # --------------------------------------------------------

    features['qty_dot_domain'] = domain.count('.')
    features['qty_hyphen_domain'] = domain.count('-')
    features['qty_underline_domain'] = domain.count('_')
    features['qty_slash_domain'] = domain.count('/')
    features['qty_questionmark_domain'] = domain.count('?')
    features['qty_equal_domain'] = domain.count('=')
    features['qty_at_domain'] = domain.count('@')
    features['qty_and_domain'] = domain.count('&')
    features['qty_exclamation_domain'] = domain.count('!')
    features['qty_space_domain'] = domain.count(' ')
    features['qty_tilde_domain'] = domain.count('~')
    features['qty_comma_domain'] = domain.count(',')
    features['qty_plus_domain'] = domain.count('+')
    features['qty_asterisk_domain'] = domain.count('*')
    features['qty_hashtag_domain'] = domain.count('#')
    features['qty_dollar_domain'] = domain.count('$')
    features['qty_percent_domain'] = domain.count('%')

    # --------------------------------------------------------
    # DOMAIN STATISTICS
    # --------------------------------------------------------

    features['qty_vowels_domain'] = sum(
        c.lower() in 'aeiou'
        for c in domain
    )

    features['domain_length'] = len(domain)

    # --------------------------------------------------------
    # DOMAIN IN IP ADDRESS
    # --------------------------------------------------------

    ip_pattern = r'^(\d{1,3}\.){3}\d{1,3}$'

    features['domain_in_ip'] = int(
        bool(re.match(ip_pattern, domain))
    )

    # --------------------------------------------------------
    # SERVER / CLIENT DOMAIN
    # --------------------------------------------------------

    features['server_client_domain'] = int(
        'server' in domain.lower()
        or 'client' in domain.lower()
    )

    # --------------------------------------------------------
    # EMAIL IN URL
    # --------------------------------------------------------

    features['email_in_url'] = int('@' in full_url)

    # --------------------------------------------------------
    # HTTPS / SSL
    # --------------------------------------------------------

    features['tls_ssl_certificate'] = int(
        parsed.scheme == 'https'
    )

    # --------------------------------------------------------
    # URL SHORTENER
    # --------------------------------------------------------

    shorteners = [
        'bit.ly',
        'tinyurl.com',
        't.co',
        'goo.gl',
        'is.gd',
        'ow.ly'
    ]

    features['url_shortened'] = int(
        any(
            shortener in domain.lower()
            for shortener in shorteners
        )
    )

    return features

LOGISTIC REGRESSION - 5-FOLD CROSS VALIDATION

Dataset shape: (9944, 56)

Features before removing constants: 42

Removing constant features:
- qty_hashtag_url
- qty_slash_domain
- qty_questionmark_domain
- qty_equal_domain
- qty_at_domain
- qty_and_domain
- qty_exclamation_domain
- qty_space_domain
- qty_tilde_domain
- qty_comma_domain
- qty_plus_domain
- qty_asterisk_domain
- qty_hashtag_domain
- qty_dollar_domain
- qty_percent_domain

Features after removing constants: 27

Final feature list:
 1. qty_dot_url
 2. qty_hyphen_url
 3. qty_underline_url
 4. qty_slash_url
 5. qty_questionmark_url
 6. qty_equal_url
 7. qty_at_url
 8. qty_and_url
 9. qty_exclamation_url
10. qty_space_url
11. qty_tilde_url
12. qty_comma_url
13. qty_plus_url
14. qty_asterisk_url
15. qty_dollar_url
16. qty_percent_url
17. length_url
18. qty_dot_domain
19. qty_hyphen_domain
20. qty_underline_domain
21. qty_vowels_domain
22. domain_length
23. domain_in_ip
24. server_client_domain
25. email_in_url
26. tls_ssl_cer

In [2]:
# ============================================================
# LOGISTIC REGRESSION MODEL
# ============================================================

import pandas as pd
import numpy as np
import re
from urllib.parse import urlparse

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# STEP 1: LOAD CLEANED DATASET
# ============================================================

print("=" * 70)
print("LOGISTIC REGRESSION BASELINE MODEL (27 FEATURES)")
print("=" * 70)

df = pd.read_csv('phishing_website_cleaned.csv')
print("\nDataset shape:", df.shape)

# ============================================================
# STEP 2: DEFINE EXACT SAME URL FEATURES AS OTHER MODELS
# ============================================================

URL_FEATURES = [
    # URL character features
    'qty_dot_url', 'qty_hyphen_url', 'qty_underline_url', 'qty_slash_url',
    'qty_questionmark_url', 'qty_equal_url', 'qty_at_url', 'qty_and_url',
    'qty_exclamation_url', 'qty_space_url', 'qty_tilde_url', 'qty_comma_url',
    'qty_plus_url', 'qty_asterisk_url', 'qty_hashtag_url', 'qty_dollar_url',
    'qty_percent_url', 'length_url',

    # Domain character features
    'qty_dot_domain', 'qty_hyphen_domain', 'qty_underline_domain', 'qty_slash_domain',
    'qty_questionmark_domain', 'qty_equal_domain', 'qty_at_domain', 'qty_and_domain',
    'qty_exclamation_domain', 'qty_space_domain', 'qty_tilde_domain', 'qty_comma_domain',
    'qty_plus_domain', 'qty_asterisk_domain', 'qty_hashtag_domain', 'qty_dollar_domain',
    'qty_percent_domain',

    # Domain statistics
    'qty_vowels_domain', 'domain_length', 'domain_in_ip', 'server_client_domain',
    'email_in_url',

    # Security / URL features
    'tls_ssl_certificate', 'url_shortened'
]

# Filter available features
available_features = [col for col in URL_FEATURES if col in df.columns]
X = df[available_features].copy()
y = df['phishing'].copy()

# ============================================================
# STEP 3: REMOVE CONSTANT FEATURES (TO GET EXACTLY 27 FEATURES)
# ============================================================

constant_features = [col for col in X.columns if X[col].nunique() <= 1]

if constant_features:
    print("\nRemoving constant features:")
    for col in constant_features:
        print("-", col)
    X = X.drop(columns=constant_features)

print("\nFeatures after removing constant features:", X.shape[1])

# ============================================================
# STEP 4: TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining set:", X_train.shape)
print("Testing set :", X_test.shape)

# ============================================================
# STEP 5: STANDARDIZATION (CRITICAL FOR LOGISTIC REGRESSION)
# ============================================================

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n✓ Standardization completed successfully.")

# ============================================================
# STEP 6: MODEL TRAINING
# ============================================================

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(X_train_scaled, y_train)

print("\n✓ Logistic Regression model trained successfully!")

# ============================================================
# STEP 7: EVALUATION
# ============================================================

y_pred = logistic_model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("\n" + "=" * 70)
print("MODEL EVALUATION RESULTS")
print("=" * 70)
print(f"Accuracy : {accuracy * 100:.2f}%")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Legitimate", "Phishing"]))

print("Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

# ============================================================
# STEP 8: FEATURE EXTRACTION FOR LIVE URL PREDICTION
# ============================================================

def extract_url_features(url):
    url = url.strip()
    if url.startswith('[') and '](' in url:
        url = url.split('](', 1)[1].rstrip(')')

    if not url.startswith(('http://', 'https://')):
        url = 'http://' + url

    parsed = urlparse(url)
    full_url = url
    domain = parsed.netloc.split(':')[0]

    features = {}

    # URL CHARACTER FEATURES
    features['qty_dot_url'] = full_url.count('.')
    features['qty_hyphen_url'] = full_url.count('-')
    features['qty_underline_url'] = full_url.count('_')
    features['qty_slash_url'] = full_url.count('/')
    features['qty_questionmark_url'] = full_url.count('?')
    features['qty_equal_url'] = full_url.count('=')
    features['qty_at_url'] = full_url.count('@')
    features['qty_and_url'] = full_url.count('&')
    features['qty_exclamation_url'] = full_url.count('!')
    features['qty_space_url'] = full_url.count(' ')
    features['qty_tilde_url'] = full_url.count('~')
    features['qty_comma_url'] = full_url.count(',')
    features['qty_plus_url'] = full_url.count('+')
    features['qty_asterisk_url'] = full_url.count('*')
    features['qty_hashtag_url'] = full_url.count('#')
    features['qty_dollar_url'] = full_url.count('$')
    features['qty_percent_url'] = full_url.count('%')
    features['length_url'] = len(full_url)

    # DOMAIN CHARACTER FEATURES
    features['qty_dot_domain'] = domain.count('.')
    features['qty_hyphen_domain'] = domain.count('-')
    features['qty_underline_domain'] = domain.count('_')
    features['qty_slash_domain'] = domain.count('/')
    features['qty_questionmark_domain'] = domain.count('?')
    features['qty_equal_domain'] = domain.count('=')
    features['qty_at_domain'] = domain.count('@')
    features['qty_and_domain'] = domain.count('&')
    features['qty_exclamation_domain'] = domain.count('!')
    features['qty_space_domain'] = domain.count(' ')
    features['qty_tilde_domain'] = domain.count('~')
    features['qty_comma_domain'] = domain.count(',')
    features['qty_plus_domain'] = domain.count('+')
    features['qty_asterisk_domain'] = domain.count('*')
    features['qty_hashtag_domain'] = domain.count('#')
    features['qty_dollar_domain'] = domain.count('$')
    features['qty_percent_domain'] = domain.count('%')

    # DOMAIN STATISTICS
    features['qty_vowels_domain'] = sum(c.lower() in 'aeiou' for c in domain)
    features['domain_length'] = len(domain)

    # DOMAIN IN IP ADDRESS
    ip_pattern = r'^(\d{1,3}\.){3}\d{1,3}$'
    features['domain_in_ip'] = int(bool(re.match(ip_pattern, domain)))

    # SERVER / CLIENT DOMAIN
    features['server_client_domain'] = int('server' in domain.lower() or 'client' in domain.lower())

    # EMAIL IN URL
    features['email_in_url'] = int('@' in full_url)

    # HTTPS / SSL
    features['tls_ssl_certificate'] = int(parsed.scheme == 'https')

    # URL SHORTENER
    shorteners = ['bit.ly', 'tinyurl.com', 't.co', 'goo.gl', 'is.gd', 'ow.ly']
    features['url_shortened'] = int(any(shortener in domain.lower() for shortener in shorteners))

    return features

LOGISTIC REGRESSION BASELINE MODEL (27 FEATURES)

Dataset shape: (9944, 56)

Removing constant features:
- qty_hashtag_url
- qty_slash_domain
- qty_questionmark_domain
- qty_equal_domain
- qty_at_domain
- qty_and_domain
- qty_exclamation_domain
- qty_space_domain
- qty_tilde_domain
- qty_comma_domain
- qty_plus_domain
- qty_asterisk_domain
- qty_hashtag_domain
- qty_dollar_domain
- qty_percent_domain

Features after removing constant features: 27

Training set: (7955, 27)
Testing set : (1989, 27)

✓ Standardization completed successfully.

✓ Logistic Regression model trained successfully!

MODEL EVALUATION RESULTS
Accuracy : 91.70%
Precision: 0.9109
Recall   : 0.8437
F1-Score : 0.8760

Classification Report:
              precision    recall  f1-score   support

  Legitimate       0.92      0.96      0.94      1298
    Phishing       0.91      0.84      0.88       691

    accuracy                           0.92      1989
   macro avg       0.92      0.90      0.91      1989
weighted a